# 04 — PPO Training with Warp + PyTorch

## 목표

03에서 만든 2048개 vectorized mass–spring cube를 PPO의 rollout environment로 사용한다.

```text
2048 Warp environments
        │
        │ zero-copy GPU state
        ▼
PyTorch Actor-Critic
        │
        │ [2048, 12] actions
        ▼
Warp physics kernels
```

CPU로 state를 복사하지 않는 것이 핵심이다.

### 상태
52차원:
- COM 기준 vertex relative position: 24
- vertex velocity: 24
- COM height: 1
- COM velocity: 3

### 행동
12개 edge spring의 rest length 조절:

\[
a \in [-1,1]^{12}
\]

### 보상

\[
r_t =
10\Delta x_{COM}
-0.01\operatorname{mean}(a^2)
-0.5\max(0,0.22-z_{COM})
\]

In [ ]:
# Colab 권장: Runtime > Change runtime type > T4 GPU
!nvidia-smi

# Warp 1.17.0의 PyPI Linux wheel은 CUDA 12.9 runtime 기반이라
# Colab의 일반적인 NVIDIA driver에서 호환성이 좋다.
%pip -q install "warp-lang==1.17.0"

import warp as wp
wp.init()
wp.print_diagnostics()

DEVICE = "cuda:0" if wp.is_cuda_available() else "cpu"
print("Selected Warp device:", DEVICE)

## 1. Physics topology

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warp as wp

SIDE = 0.4

def make_cube_topology(side=SIDE, z0=0.45):
    # bit pattern 순서의 8 vertices
    verts = np.array([
        [0,0,0], [1,0,0], [0,1,0], [1,1,0],
        [0,0,1], [1,0,1], [0,1,1], [1,1,1],
    ], dtype=np.float32)

    verts *= side
    verts[:, 0] -= side * 0.5
    verts[:, 1] -= side * 0.5
    verts[:, 2] += z0

    springs = []
    actuator_index = []
    actuator_count = 0

    # 모든 8C2 = 28 pair를 연결:
    # 12 edge + 12 face diagonal + 4 body diagonal
    for i in range(8):
        for j in range(i + 1, 8):
            d = np.linalg.norm(verts[j] - verts[i])
            springs.append((i, j))

            if np.isclose(d, side, atol=1e-5):
                actuator_index.append(actuator_count)
                actuator_count += 1
            else:
                actuator_index.append(-1)

    spring_i = np.array([s[0] for s in springs], dtype=np.int32)
    spring_j = np.array([s[1] for s in springs], dtype=np.int32)
    rest = np.array(
        [np.linalg.norm(verts[j] - verts[i]) for i, j in springs],
        dtype=np.float32
    )

    return verts, spring_i, spring_j, rest, np.array(actuator_index, np.int32)

BASE_X, SPRING_I, SPRING_J, BASE_REST, ACT_IDX = make_cube_topology()

print("particles:", len(BASE_X))
print("springs:", len(SPRING_I))
print("actuated edge springs:", np.sum(ACT_IDX >= 0))

## 2. Vectorized Warp physics

In [ ]:
@wp.kernel
def reset_vec(
    base_x: wp.array(dtype=wp.vec3),
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    env_spacing: float,
):
    tid = wp.tid()
    env = tid // 8
    p = tid - env * 8

    # visualization/debug 시 서로 다른 위치에 놓고 싶으면 y offset을 줄 수 있다.
    # RL 상태에서는 translation invariance를 쓰므로 기본은 같은 위치에 겹쳐 둔다.
    x[tid] = base_x[p]
    v[tid] = wp.vec3(0.0, 0.0, 0.0)


@wp.kernel
def compute_force_vec(
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    spring_i: wp.array(dtype=wp.int32),
    spring_j: wp.array(dtype=wp.int32),
    rest: wp.array(dtype=float),
    actuator_index: wp.array(dtype=wp.int32),
    actions: wp.array2d(dtype=float),
    force: wp.array(dtype=wp.vec3),
    n_springs: int,
    mass: float,
    k: float,
    c: float,
    act_amp: float,
    ground_k: float,
    ground_c: float,
    friction: float,
):
    tid = wp.tid()
    env = tid // 8
    p = tid - env * 8
    base = env * 8

    xp = x[tid]
    vp = v[tid]
    f = wp.vec3(0.0, 0.0, -9.81 * mass)

    for s in range(n_springs):
        i = spring_i[s]
        j = spring_j[s]

        if p == i or p == j:
            xi = x[base + i]
            xj = x[base + j]
            vi = v[base + i]
            vj = v[base + j]

            d = xj - xi
            L = wp.length(d)
            n = d / (L + 1.0e-8)

            L0 = rest[s]
            aidx = actuator_index[s]
            if aidx >= 0:
                L0 = L0 * (1.0 + act_amp * actions[env, aidx])

            rel = wp.dot(vj - vi, n)
            mag = k * (L - L0) + c * rel
            fs = mag * n

            if p == i:
                f = f + fs
            else:
                f = f - fs

    if xp[2] < 0.0:
        f = f + wp.vec3(0.0, 0.0, -ground_k * xp[2])

        if vp[2] < 0.0:
            f = f + wp.vec3(0.0, 0.0, -ground_c * vp[2])

        f = f + wp.vec3(-friction * vp[0], -friction * vp[1], 0.0)

    force[tid] = f


@wp.kernel
def integrate_vec(
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    force: wp.array(dtype=wp.vec3),
    mass: float,
    dt: float,
):
    tid = wp.tid()

    a = force[tid] / mass
    v_new = v[tid] + a * dt
    x_new = x[tid] + v_new * dt

    v[tid] = v_new
    x[tid] = x_new

In [ ]:
import torch

TORCH_DEVICE = torch.device("cuda" if DEVICE.startswith("cuda") else "cpu")

class VectorizedWarpCube:
    def __init__(
        self,
        num_envs=2048,
        device=DEVICE,
        dt=0.0025,
        substeps=4,
        mass=0.15,
        k=180.0,
        c=2.0,
        act_amp=0.20,
        ground_k=1500.0,
        ground_c=15.0,
        friction=2.0,
    ):
        self.num_envs = num_envs
        self.device = device
        self.dt = dt
        self.substeps = substeps
        self.mass = mass
        self.k = k
        self.c = c
        self.act_amp = act_amp
        self.ground_k = ground_k
        self.ground_c = ground_c
        self.friction = friction

        self.n_particles = 8
        self.n_springs = len(SPRING_I)
        self.n_act = int(np.sum(ACT_IDX >= 0))

        self.base_x = wp.array(BASE_X, dtype=wp.vec3, device=device)
        self.spring_i = wp.array(SPRING_I, dtype=wp.int32, device=device)
        self.spring_j = wp.array(SPRING_J, dtype=wp.int32, device=device)
        self.rest = wp.array(BASE_REST, dtype=float, device=device)
        self.actuator_index = wp.array(ACT_IDX, dtype=wp.int32, device=device)

        N = num_envs * 8
        self.x = wp.zeros(N, dtype=wp.vec3, device=device)
        self.v = wp.zeros(N, dtype=wp.vec3, device=device)
        self.force = wp.zeros(N, dtype=wp.vec3, device=device)

        self.reset()

    def reset(self):
        wp.launch(
            reset_vec,
            dim=self.num_envs * 8,
            inputs=[self.base_x, self.x, self.v, 0.0],
            device=self.device,
        )

    def state_torch(self):
        # zero-copy view: [E*8,3] -> [E,8,3]
        x_t = wp.to_torch(self.x).view(self.num_envs, 8, 3)
        v_t = wp.to_torch(self.v).view(self.num_envs, 8, 3)
        return x_t, v_t

    def observation(self):
        x_t, v_t = self.state_torch()

        com = x_t.mean(dim=1)
        com_v = v_t.mean(dim=1)
        rel = x_t - com[:, None, :]

        obs = torch.cat(
            [
                rel.reshape(self.num_envs, -1),   # 24
                v_t.reshape(self.num_envs, -1),   # 24
                com[:, 2:3],                      # 1
                com_v,                            # 3
            ],
            dim=1,
        )
        return obs

    def step(self, actions_t):
        # actions_t: [E, 12], same GPU device, contiguous float32
        actions_t = actions_t.contiguous().to(dtype=torch.float32)
        actions_w = wp.from_torch(actions_t, requires_grad=False)

        x_before, _ = self.state_torch()
        com_x_before = x_before[:, :, 0].mean(dim=1).clone()

        for _ in range(self.substeps):
            wp.launch(
                compute_force_vec,
                dim=self.num_envs * 8,
                inputs=[
                    self.x, self.v,
                    self.spring_i, self.spring_j,
                    self.rest, self.actuator_index,
                    actions_w, self.force,
                    self.n_springs,
                    self.mass, self.k, self.c, self.act_amp,
                    self.ground_k, self.ground_c, self.friction,
                ],
                device=self.device,
            )
            wp.launch(
                integrate_vec,
                dim=self.num_envs * 8,
                inputs=[self.x, self.v, self.force, self.mass, self.dt],
                device=self.device,
            )

        x_after, _ = self.state_torch()
        com = x_after.mean(dim=1)

        dx = com[:, 0] - com_x_before
        energy = actions_t.square().mean(dim=1)
        low_height = torch.relu(0.22 - com[:, 2])

        reward = 10.0 * dx - 0.01 * energy - 0.5 * low_height
        obs = self.observation()

        return obs, reward, {
            "com_x": com[:, 0],
            "com_z": com[:, 2],
            "energy": energy,
        }

## 3. Actor–Critic

연속 행동이므로 Gaussian policy를 사용하고,
샘플한 값을 `tanh`로 squashing해 `[-1,1]` 범위로 제한한다.

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Normal

class ActorCritic(nn.Module):
    def __init__(self, obs_dim=52, act_dim=12, hidden=128):
        super().__init__()

        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )

        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

        self.log_std = nn.Parameter(torch.full((act_dim,), -0.5))

    def distribution(self, obs):
        mu = self.actor(obs)
        std = self.log_std.exp().expand_as(mu)
        return Normal(mu, std)

    def value(self, obs):
        return self.critic(obs).squeeze(-1)

    @torch.no_grad()
    def sample(self, obs):
        d = self.distribution(obs)
        z = d.sample()
        action = torch.tanh(z)

        # tanh squashing correction
        logp = d.log_prob(z).sum(-1)
        logp -= torch.log(1.0 - action.square() + 1e-6).sum(-1)

        value = self.value(obs)
        return action, z, logp, value

    def evaluate(self, obs, z):
        d = self.distribution(obs)
        action = torch.tanh(z)

        logp = d.log_prob(z).sum(-1)
        logp -= torch.log(1.0 - action.square() + 1e-6).sum(-1)

        entropy = d.entropy().sum(-1)
        value = self.value(obs)
        return logp, entropy, value

## 4. Generalized Advantage Estimation

\[
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
\]

\[
A_t = \delta_t + \gamma\lambda A_{t+1}
\]

In [ ]:
@torch.no_grad()
def compute_gae(rewards, values, last_value, gamma=0.99, lam=0.95):
    # rewards: [T,E]
    # values:  [T,E]
    T, E = rewards.shape
    adv = torch.zeros_like(rewards)
    gae = torch.zeros(E, device=rewards.device)

    for t in reversed(range(T)):
        next_value = last_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_value - values[t]
        gae = delta + gamma * lam * gae
        adv[t] = gae

    returns = adv + values
    return adv, returns

## 5. PPO update

\[
r_t(	heta) =
rac{\pi_	heta(a_t|s_t)}
     {\pi_{	heta_{old}}(a_t|s_t)}
\]

\[
L^{CLIP}
=
\mathbb{E}
[
\min(
r_tA_t,
\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t
)
]
\]

### Colab 권장 설정

빠른 수업 데모:
- `num_envs=512`
- `updates=10`
- `horizon=32`

본 실습:
- `num_envs=2048`
- `updates=40~100`
- `horizon=32~64`

GPU memory가 부족하면 `num_envs` 또는 `minibatch`를 줄인다.

In [ ]:
def train_ppo(
    num_envs=2048,
    updates=40,
    horizon=32,
    ppo_epochs=4,
    minibatch=8192,
    lr=3e-4,
    gamma=0.99,
    lam=0.95,
    clip_eps=0.2,
    vf_coef=0.5,
    ent_coef=0.002,
    grad_clip=0.5,
):
    sim = VectorizedWarpCube(num_envs=num_envs)
    obs = sim.observation()

    policy = ActorCritic(
        obs_dim=obs.shape[1],
        act_dim=sim.n_act,
        hidden=128
    ).to(TORCH_DEVICE)

    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)

    history = {
        "mean_rollout_reward": [],
        "mean_com_x": [],
        "policy_loss": [],
        "value_loss": [],
        "entropy": [],
    }

    # 각 update 시작 시 같은 초기 상태에서 비교하기 쉽게 reset
    # 더 어려운 continuous locomotion 과제로 확장하려면 episode bookkeeping을 추가하면 된다.
    for update in range(updates):
        sim.reset()
        obs = sim.observation()

        obs_buf = []
        z_buf = []
        logp_buf = []
        reward_buf = []
        value_buf = []

        for t in range(horizon):
            action, z, logp, value = policy.sample(obs)
            next_obs, reward, info = sim.step(action)

            obs_buf.append(obs)
            z_buf.append(z)
            logp_buf.append(logp)
            reward_buf.append(reward)
            value_buf.append(value)

            obs = next_obs

        with torch.no_grad():
            last_value = policy.value(obs)

        O = torch.stack(obs_buf)          # [T,E,obs]
        Z = torch.stack(z_buf)            # [T,E,act]
        OLD_LOGP = torch.stack(logp_buf)  # [T,E]
        R = torch.stack(reward_buf)       # [T,E]
        V = torch.stack(value_buf)        # [T,E]

        ADV, RET = compute_gae(R, V, last_value, gamma=gamma, lam=lam)

        # flatten T,E -> batch
        O = O.reshape(-1, O.shape[-1])
        Z = Z.reshape(-1, Z.shape[-1])
        OLD_LOGP = OLD_LOGP.reshape(-1)
        ADV = ADV.reshape(-1)
        RET = RET.reshape(-1)

        ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)

        B = O.shape[0]
        last_pl = last_vl = last_ent = 0.0

        for epoch in range(ppo_epochs):
            perm = torch.randperm(B, device=TORCH_DEVICE)

            for start in range(0, B, minibatch):
                idx = perm[start:start + minibatch]

                new_logp, entropy, value = policy.evaluate(O[idx], Z[idx])

                ratio = torch.exp(new_logp - OLD_LOGP[idx])
                unclipped = ratio * ADV[idx]
                clipped = torch.clamp(
                    ratio,
                    1.0 - clip_eps,
                    1.0 + clip_eps
                ) * ADV[idx]

                policy_loss = -torch.minimum(unclipped, clipped).mean()
                value_loss = (value - RET[idx]).square().mean()
                entropy_mean = entropy.mean()

                loss = (
                    policy_loss
                    + vf_coef * value_loss
                    - ent_coef * entropy_mean
                )

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), grad_clip)
                optimizer.step()

                last_pl = float(policy_loss.detach())
                last_vl = float(value_loss.detach())
                last_ent = float(entropy_mean.detach())

        mean_reward = float(R.mean())
        mean_com_x = float(info["com_x"].mean())

        history["mean_rollout_reward"].append(mean_reward)
        history["mean_com_x"].append(mean_com_x)
        history["policy_loss"].append(last_pl)
        history["value_loss"].append(last_vl)
        history["entropy"].append(last_ent)

        print(
            f"update {update+1:03d}/{updates} | "
            f"reward {mean_reward:+.5f} | "
            f"COM x {mean_com_x:+.4f} | "
            f"entropy {last_ent:.3f}"
        )

    return policy, history

## 6. 먼저 512 env smoke test

In [ ]:
# 전체 pipeline이 정상인지 빠르게 확인
smoke_policy, smoke_hist = train_ppo(
    num_envs=512,
    updates=3,
    horizon=16,
    ppo_epochs=2,
    minibatch=2048,
)

## 7. 2048 env 본 학습

In [ ]:
# 수업 중에는 updates=20 정도로 시작하고,
# 과제/실험에서는 40~100 이상을 권장.
policy, hist = train_ppo(
    num_envs=2048,
    updates=40,
    horizon=32,
    ppo_epochs=4,
    minibatch=8192,
)

## 8. Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(hist["mean_rollout_reward"])
ax.set_xlabel("PPO update")
ax.set_ylabel("mean step reward")
ax.set_title("Training reward")
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(hist["mean_com_x"])
ax.set_xlabel("PPO update")
ax.set_ylabel("mean final COM x")
ax.set_title("Forward displacement")
ax.grid(alpha=0.3)
plt.show()

## 9. Deterministic rollout

In [ ]:
@torch.no_grad()
def evaluate_policy(policy, steps=300):
    sim = VectorizedWarpCube(num_envs=1)
    sim.reset()
    obs = sim.observation()

    frames = []
    com_x = []
    rewards = []

    for t in range(steps):
        x_t, _ = sim.state_torch()
        frames.append(x_t[0].detach().cpu().numpy().copy())

        # deterministic mean action
        action = torch.tanh(policy.actor(obs))
        obs, reward, info = sim.step(action)

        rewards.append(float(reward[0]))
        com_x.append(float(info["com_x"][0]))

    return np.array(frames), np.array(com_x), np.array(rewards)

In [ ]:
frames, com_x, rewards = evaluate_policy(policy, steps=300)

print("initial COM x:", frames[0,:,0].mean())
print("final COM x:", frames[-1,:,0].mean())
print("displacement:", frames[-1,:,0].mean() - frames[0,:,0].mean())

plt.figure(figsize=(8,3))
plt.plot(com_x)
plt.xlabel("step")
plt.ylabel("COM x")
plt.title("Learned policy rollout")
plt.grid(alpha=0.3)
plt.show()

## 10. 시작/끝 자세

In [ ]:
def plot_frame(x, ax, title):
    for i, j in zip(SPRING_I, SPRING_J):
        p, q = x[i], x[j]
        ax.plot([p[0],q[0]], [p[1],q[1]], [p[2],q[2]], alpha=0.3)
    ax.scatter(x[:,0], x[:,1], x[:,2], s=45)
    ax.set_title(title)
    ax.set_zlim(-0.05, 0.9)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

fig = plt.figure(figsize=(11,4))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")
plot_frame(frames[0], ax1, "start")
plot_frame(frames[-1], ax2, "end")
plt.show()

# 실험 과제

## A. Reward ablation
아래 항목을 각각 제거하고 학습 결과를 비교한다.

1. energy penalty
2. low-height penalty
3. forward reward scaling

## B. Physics ablation

- damping `c ∈ {0.5, 2.0, 5.0}`
- stiffness `k ∈ {100, 180, 300}`
- actuation amplitude `A ∈ {0.1, 0.2, 0.35}`

## C. Observation ablation

velocity 24차원을 제거해보라.

질문:
- 학습이 느려지는가?
- 이 상태가 여전히 Markov state라고 볼 수 있는가?

## D. 구조 실험

28개 spring 대신:
- 12 edge only
- edge + face diagonal
- all 28

을 비교하라.

## E. 보고서에 포함할 것

- seed 3개 이상의 learning curve
- 최종 COM displacement
- energy consumption
- learned gait 설명
- 실패한 설정 하나와 실패 원인 분석

# 확장 프로젝트

1. **2D navigation**  
   target direction을 observation에 추가.

2. **Terrain randomization**  
   friction / ground height를 episode마다 변경.

3. **Domain randomization**  
   mass, stiffness, damping을 randomize해 robust controller 학습.

4. **Morphology learning**  
   spring stiffness/rest length 자체를 outer-loop optimization.

5. **Differentiable simulation**  
   Warp의 autodiff와 PPO를 비교해 직접 trajectory optimization 수행.